In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
from pathlib import Path
import logging
import seaborn as sns


RANDOM_STATE = 42

In [59]:

# Загрузка датасета
DATASET_URL = "https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/train_data.csv"
data = pd.read_csv(DATASET_URL)
print(f"Размер: {data.shape}")
data.head()


Размер: (6200, 48)


,id,atm_group,address_raw,address_geocoded,geo_lon,geo_lat,country,region,municipality,city,...,nearest_public_transport_dist_m,count_public_transport_300m,nearest_parking_dist_m,count_parking_300m,nearest_education_dist_m,count_education_300m,nearest_subway_dist_m,nearest_post_offices_dist_m,count_post_offices_300m,has_subway_nearby
0,5.0,496.5,BUDENNOGO 7A ELISTA,"Россия, Республика Калмыкия, Элиста, улица С.М...",44.260605,46.318231,Россия,Республика Калмыкия,городской округ Элиста,Элиста,...,93.6,5,143.7,3,247.3,1,0.0,0.0,0,False
1,6.0,496.5,"HO CHI MIHN AVE, 19 ULYANOVSK","Россия, Ульяновск, проспект Хо Ши Мина, 19",48.300652,54.270443,Россия,Ульяновская область,городской округ Ульяновск,Ульяновск,...,89.9,6,NaN,0,260.8,1,0.0,220.4,2,False
2,7.0,496.5,SHELESTA 116A KHABAROVSK,"Россия, Хабаровск, улица Шелеста, 116А",135.052594,48.520497,Россия,Хабаровский край,городской округ Хабаровск,Хабаровск,...,33.8,8,112.9,6,0.0,0,0.0,186.6,2,False
3,8.0,496.5,ORDZHONIKIDZE 52 YAKUTSK,"Россия, Республика Саха (Якутия), Якутск, улиц...",129.721308,62.025566,Россия,Республика Саха (Якутия),городской округ Якутск,Якутск,...,119.8,7,246.9,4,195.4,5,0.0,167.2,1,False
4,10.0,496.5,"VETERANOV AVE, 3 KRASNOKAMENS","Россия, Забайкальский край, Краснокаменск, про...",118.027480,50.090714,Россия,Забайкальский край,Краснокаменский муниципальный округ,Краснокаменск,...,70.6,3,48.3,5,NaN,0,NaN,NaN,0,False


## EDA 

In [72]:
# Конфигурация на основе датасета
EXCLUDE_COLS = [
    "id", "address_raw", "address_geocoded", "street", "house", 
    "geo_lon", "geo_lat", "municipality", "country", "atm_group" # ПОКА ВЫКИНУЛА atm_group из обучения 
]

BIN_FEATURES = [
    "is_24_7", "contactless_tech", "qr_codes", "usd_available", "eur_available",
    "cash_in", "cash_out", "cashless_pay", "account_statement", 
    "access_for_disabled", "transfer_p2p", "transfer_a2a", "loan_payments", 
    "has_subway_nearby"
]

LOCATION_FEATURES = [
    "population_density_per_km2", "nearest_malls_dist_m", "count_malls_300m",
    "nearest_supermarkets_dist_m", "nearest_pharmacies_hospitals_dist_m",
    "count_pharmacies_hospitals_300m", "count_banks_atms_300m",
    "nearest_cafes_dist_m", "count_cafes_300m", "nearest_restaurants_dist_m",
    "count_restaurants_300m", "nearest_public_transport_dist_m",
    "count_public_transport_300m", "nearest_parking_dist_m", "count_parking_300m",
    "nearest_education_dist_m", "count_education_300m", "nearest_subway_dist_m",
    "nearest_post_offices_dist_m"
]

CATEGORICAL_FEATURES = ["city"]
TARGET = "target"

print(f"Бинарных фичей: {len(BIN_FEATURES)}")
print(f"Локационных фичей: {len(LOCATION_FEATURES)}")


Бинарных фичей: 14
Локационных фичей: 19


## INPUT DATASET vs STANDART COLUMNS FOR PYPLINE

In [ ]:
class RegionGrouper(BaseEstimator, TransformerMixin):
   
    def __init__(self, rare_threshold=0.01):
        self.rare_threshold = rare_threshold
        self.rare_regions_ = None
    
    def fit(self, X, y=None):
        df = pd.DataFrame(X)
        if 'region' in df.columns:
            cnt = df['region'].value_counts()
            self.rare_regions_ = set(cnt[cnt < self.rare_threshold * len(df)].index)
        return self
    
    def transform(self, X):
        df = pd.DataFrame(X)
        if 'region' in df.columns and self.rare_regions_ is not None:
            df['region_grouped'] = df['region'].apply(
                lambda x: '__OTHER__' if pd.isna(x) or x in self.rare_regions_ else str(x)
            )
            df.drop(columns=['region'], inplace=True, errors='ignore')
        return df

class BinaryToFloat(BaseEstimator, TransformerMixin):
    def transform(self, X):
        df = pd.DataFrame(X)
        available_bin = [col for col in BIN_FEATURES if col in df.columns]
        for col in available_bin:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(float)
        return df
    def fit(self, X, y=None): return self

class ATMLocationFeatures(BaseEstimator, TransformerMixin):
    def transform(self, X):
        df = pd.DataFrame(X)
        dist_cols = [col for col in df.columns if 'dist_m' in col]
        for col in dist_cols:
            df[f'{col}_log'] = np.log1p(df[col].fillna(0))
        count_cols = [col for col in df.columns if 'count_' in col]
        if count_cols:
            df['poi_density_300m'] = df[count_cols].sum(axis=1)
        return df
    def fit(self, X, y=None): return self


In [79]:


# Доступные колонки из наших списков
available_location = [col for col in LOCATION_FEATURES if col in data.columns]
available_bin = [col for col in BIN_FEATURES if col in data.columns]
available_cat = [col for col in CATEGORICAL_FEATURES if col in data.columns]

print(f"\nДоступно:")
print(f"   Локационные: {len(available_location)}/{len(LOCATION_FEATURES)}")
print(f"   Бинарные:     {len(available_bin)}/{len(BIN_FEATURES)}")
print(f"   Категории:    {len(available_cat)}/{len(CATEGORICAL_FEATURES)}")

# Безопасная подготовка X
exclude_cols_present = [col for col in EXCLUDE_COLS if col in data.columns]
X = data.drop(columns=exclude_cols_present + [TARGET], errors='ignore')





Доступно:
   Локационные: 19/19
   Бинарные:     14/14
   Категории:    1/1


## Preprocessor


In [80]:
def create_preprocessor(X):

    
    all_cols = X.columns.tolist()
    available_num = [col for col in LOCATION_FEATURES if col in all_cols]
    available_bin = [col for col in BIN_FEATURES if col in all_cols]
    
    print(f"НАЙДЕНЫ (БЕЗ atm_group):")
    print(f"num: {len(available_num)}")
    print(f"bin: {len(available_bin)}")
    

    initial_pipeline = Pipeline([
        ('region_grouper', RegionGrouper()),  # region → region_grouped
        ('binary_to_float', BinaryToFloat()),
        ('location_features', ATMLocationFeatures())
    ])
    
  
    transformers = [
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), available_num),
        ('bin', SimpleImputer(strategy='most_frequent'), available_bin),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
        ]), CATEGORICAL_FEATURES)  # Только ['city']
    ]
    
    preprocessor = ColumnTransformer(transformers, remainder='drop')
    

    full_pipeline = Pipeline([
        ('initial', initial_pipeline),
        ('preprocessor', preprocessor),
        ('final_scaler', StandardScaler())
    ])
    
    return full_pipeline

full_pipeline = create_preprocessor(X)


НАЙДЕНЫ (БЕЗ atm_group):
num: 19
bin: 14


## Pypline Preprocessor Validate and Safe

In [ ]:

full_pipeline.fit(X)

print("Тест:")
X_processed = full_pipeline.transform(X[:100])
print(f"Shape: {X_processed.shape}")
print(f"NA: {np.isnan(X_processed).sum()}")
print(f"Finite: {np.isfinite(X_processed).all()}")


import os
from pathlib import Path

MODEL_DIR = Path("/Users/aleks_not007/Desktop/HSE_mag/year_project/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)  # Создаем если нет

save_path = MODEL_DIR / "preprocessor.pkl"

try:
    joblib.dump(full_pipeline, save_path)
    print(f"✅ СОХРАНЕНО: {save_path}")
    print(f"Папка: {MODEL_DIR}")
    print(f"Размер файла: {save_path.stat().st_size / 1024:.1f} KB")
except Exception as e:
    print(f"❌ Ошибка: {e}")


Тест:
✅ Shape: (100, 802)
✅ NA: 0
✅ Finite: True
✅ СОХРАНЕНО: /Users/aleks_not007/Desktop/HSE_mag/year_project/model/preprocessor.pkl
Папка: /Users/aleks_not007/Desktop/HSE_mag/year_project/model
Размер файла: 48.6 KB


/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [ ]:
# === Разделение данных на признаки и целевую переменную ===
# === Удаляем строки с пропущенными значениями ===
# data = data.dropna()

# === Удаляем координаты ===
X = data.drop(columns=['target', 'geo_lon', 'geo_lat'])
y = data['target']

# === Кодируем категориальный признак region ===
X = pd.get_dummies(X, columns=['region'], drop_first=True)

# === Теперь все признаки числовые ===
X = X.select_dtypes(include=['int64', 'float64', 'uint8'])

# === Базовый сплит данных (25% тест, seed = 42) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


NameError: name 'data' is not defined

## Train

In [83]:
import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [90]:
# === Разделение данных на признаки и целевую переменную ===
# === Удаляем строки с пропущенными значениями ===
# === Удаляем координаты ===
X = data.drop(columns=['target'])
y = data['target']

# === Базовый сплит данных (25% тест, seed = 42) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Загружаем обученный preprocessor
MODEL_DIR = Path("/Users/aleks_not007/Desktop/HSE_mag/year_project/model")
preprocessor = joblib.load(MODEL_DIR / "preprocessor.pkl")

print(f"X shape: {X_train.shape}")




X shape: (2640, 47)


In [91]:
from sklearn.pipeline import Pipeline

models = {
    'LinearRegression': LinearRegression(),
    'LassoCV': LassoCV(cv=5, random_state=42),
    'RidgeCV': RidgeCV(cv=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42, max_depth=10)
}

results = {}

for name, base_model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),  # работает по сырым X
        ('model', base_model),
    ])
    
    cv_scores = cross_val_score(
        pipe,
        X_train,  # СЫРЫЕ признаки, без .transform
        y_train,
        cv=5,
        scoring='neg_root_mean_squared_error'
    )
    
    results[name] = {
        'CV_RMSE_mean': -cv_scores.mean(),
        'CV_RMSE_std': cv_scores.std(),
        'CV_scores': cv_scores
    }
    

/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-pac

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

# Таблица результатов
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('CV_RMSE_mean')
results_df.rename(columns={
    'CV_RMSE_mean': 'rmse_mean',
    'CV_RMSE_std': 'rmse_std'
}, inplace=True)
display(results_df)

best_model_name = results_df.index[0]
print(f"ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")

best_base_model = models[best_model_name]
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_base_model),
])

# Обучаем на train
best_pipeline.fit(X_train, y_train)

# Оценка на test
y_pred_test = best_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(mse)

print(f"\n{best_model_name}")
print(f"   CV RMSE (mean): {results_df.loc[best_model_name, 'rmse_mean']:.4f}")
print(f"   CV RMSE (std):  {results_df.loc[best_model_name, 'rmse_std']:.4f}")
print(f"   Test RMSE:      {test_rmse:.4f}")


,rmse_mean,rmse_std,CV_scores
LassoCV,0.066698,0.002133,"[-0.06799632584676335, -0.0687109478448905, -0..."
RidgeCV,0.067568,0.00208,"[-0.06970467504395247, -0.06883160902169282, -..."
LinearRegression,0.068208,0.001416,"[-0.06970975286897411, -0.06904791918484743, -..."
DecisionTree,0.07145,0.002658,"[-0.07224649307974941, -0.07527481787877256, -..."


🏆 ЛУЧШАЯ МОДЕЛЬ: LassoCV

🏆 LassoCV
   CV RMSE (mean): 0.0667
   CV RMSE (std):  0.0021
   Test RMSE:      0.0675


/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-pac

In [98]:
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

test_results = {}

for name, base_model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', base_model),
    ])
    
    # Обучаем на train
    pipe.fit(X_train, y_train)
    
    # Предсказываем на test
    y_pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    test_results[name] = {
        'test_rmse': rmse
    }
    print(f"{name:15} | Test RMSE: {rmse:.4f}")

test_results_df = pd.DataFrame(test_results).T.sort_values('test_rmse')
display(test_results_df)


/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-pac

LinearRegression | Test RMSE: 0.0689


/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-pac

LassoCV         | Test RMSE: 0.0675
RidgeCV         | Test RMSE: 0.0683
DecisionTree    | Test RMSE: 0.0712


/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-pac

,test_rmse
LassoCV,0.067453
RidgeCV,0.068339
LinearRegression,0.068864
DecisionTree,0.071246


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np
import joblib
from pathlib import Path

best_base_model = models[best_model_name]

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),  
    ('model', best_base_model),
])

# Обучаем на ВСЕХ данных 
final_pipeline.fit(X, y)

# RMSE на всём датасете (in-sample)
y_pred_all = final_pipeline.predict(X)
mse_all = mean_squared_error(y, y_pred_all)
rmse_all = np.sqrt(mse_all)
print(f"RMSE на FULL датасете: {rmse_all:.4f}")

# Путь сохранения
MODEL_DIR = Path("/Users/aleks_not007/Desktop/HSE_mag/year_project/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
final_path = MODEL_DIR / "final_atm_pipeline.pkl"

# Сохраняем ПОЛНЫЙ пайплайн (preprocessor + model)
joblib.dump(final_pipeline, final_path)
print(f"\nФИНАЛЬНЫЙ ПАЙПЛАЙН СОХРАНЁН: {final_path}")


📊 RMSE на всём датасете: 0.0632

ФИНАЛЬНЫЙ ПАЙПЛАЙН СОХРАНЁН: /Users/aleks_not007/Desktop/HSE_mag/year_project/model/final_atm_pipeline.pkl


/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [ ]:
# test_model_load.py 
model = joblib.load("УКАЗАТЬ ПУТЬ К МЕСТУУ ХРАНЕНИЯ ПАЙПЛАЙНА")
print("Steps:", model.named_steps.keys())
print("Model type:", type(model.named_steps["model"]))

sample = pd.DataFrame([{
    "id": 5.0,
    "atm_group": 496.5,
    "address_raw": "BUDENNOGO 7A              ELISTA      ",
    "address_geocoded": "Россия, Республика Калмыкия, Элиста, улица С.М. Будённого, 7А",
    "geo_lon": 44.260605,
    "geo_lat": 46.318231,
    "country": "Россия",
    "region": "Республика Калмыкия",
    "municipality": "городской округ Элиста",
    "city": "Элиста",
    "street": "улица С.М. Будённого",
    "house": "7А",
    "population_density_per_km2": 3.52,
    "is_24_7": False,
    "contactless_tech": False,
    "qr_codes": False,
    "usd_available": False,
    "eur_available": False,
    "cash_in": True,
    "cash_out": True,
    "cashless_pay": False,
    "account_statement": True,
    "access_for_disabled": True,
    "transfer_p2p": True,
    "transfer_a2a": False,
    "loan_payments": False,
    "nearest_malls_dist_m": 928.7,
    "count_malls_300m": 0,
    "nearest_supermarkets_dist_m": 364.3,
    "count_supermarkets_300m": 0,
    "nearest_pharmacies_hospitals_dist_m": 242.6,
    "count_pharmacies_hospitals_300m": 2,
    "count_banks_atms_300m": 2,
    "nearest_cafes_dist_m": 124.5,
    "count_cafes_300m": 1,
    "nearest_restaurants_dist_m": 317.9,
    "count_restaurants_300m": 0,
    "nearest_public_transport_dist_m": 93.6,
    "count_public_transport_300m": 5,
    "nearest_parking_dist_m": 143.7,
    "count_parking_300m": 3,
    "nearest_education_dist_m": 247.3,
    "count_education_300m": 1,
    "nearest_subway_dist_m": 0.0,
    "nearest_post_offices_dist_m": 0.0,
    "count_post_offices_300m": 0,
    "has_subway_nearby": False,
}])

pred = model.predict(sample)
print(pred[0])


Steps: dict_keys(['preprocessor', 'model'])
Model type: <class 'sklearn.linear_model._coordinate_descent.LassoCV'>
0.010866362851978548


/Users/aleks_not007/Library/Python/3.9/lib/python/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [ ]:
# app/core/model.py
from pathlib import Path
from typing import Dict, Any
import joblib
import pandas as pd

MODEL_PATH = Path("УКАЗАТЬ ПУТЬ К МЕСТУУ ХРАНЕНИЯ ПАЙПЛАЙНА")

class ATMModelService:
    def __init__(self, model_path: Path = MODEL_PATH):
        self.model_path = model_path
        self.model = self._load_model()

    def _load_model(self):
        if not self.model_path.exists():
            raise FileNotFoundError(f"Model not found: {self.model_path}")
        return joblib.load(self.model_path)

    def predict_popularity(self, features: Dict[str, Any]) -> float:
        df = pd.DataFrame([features])
        y_pred = self.model.predict(df)[0]
        return float(y_pred)

# singleton для использования в FastAPI
atm_model_service = ATMModelService()
